In [1]:
def readctm(filename):
    lines = []
    with open(filename) as inf:
        for line in inf.readlines():
            lines.append(line.strip().split(" "))
    return lines

In [37]:
def strctm(text):
    return [line.strip().split(" ") for line in text.split("\n") if line != ""]

In [2]:
def normword(word):
    if word.endswith("..."):
        word = word[0:-3]
    if word[-1:] in ".,?!-:;":
        word = word[0:-1]
    return word.lower()

In [32]:
def push_back_ins(ctm_lines, start, end, offset):
    pre = []
    post = []
    buf = []

    for line in ctm_lines:
        if float(line[2]) < start:
            pre.append(line)
        elif float(line[2]) <= end:
            buf.append(line)
        else:
            post.append(line)

    i = 0
    while i < (len(buf) - offset):
        print(buf[i], buf[i+offset])
        replace = buf[i+offset][6]
        replacenorm = normword(replace)
        buf[i][4] = replacenorm
        buf[i][6] = replace
        buf[i][7] = "cor"
        i += 1
    while i < len(buf):
        buf[i][6] = "-"
        buf[i][7] = "-"
        i += 1
        
    return pre + buf + post

In [56]:
def merge_left(ctmlines, replacement = None):
    outword = ctmlines[-1][6]
    if replacement is not None:
        outword = replacement
    out = ctmlines[0][0:3]
    out += [ctmlines[-1][3], normword(outword), "1.0", outword, "cor"]
    return out

In [33]:
ctmlines = readctm("/tmp/ctmedit/hsi_1_0515_209_001_inter.txt")

In [60]:
def printctm(toprint):
    for ll in toprint:
        print(" ".join(ll))

In [67]:
def as_csv(c):
    print(f"{c[2]}\t{c[3]}\t{c[6]}")

In [76]:
from pathlib import Path

CTMDIR = Path("/Users/joregan/Playing/hsi_ctmedit")

for ctmfile in CTMDIR.glob("*.txt"):
    ctmlines = readctm(str(ctmfile))
    outlines = []
    for ctmline in ctmlines:
        if len(ctmline) < 7:
            outlines.append(ctmline)
            continue
        if ctmline[7] != "sub":
            outlines.append(ctmline)
        else:
            if ctmline[4] == normword(ctmline[6]):
                ctmline[7] = "cor"
            outlines.append(ctmline)
    with open(str(ctmfile), "w") as outf:
        for line in outlines:
            outf.write(" ".join(line) + "\n")

80008 cor
22110 sub
11758 ins
7578 del

In [75]:
from pathlib import Path

CTMDIR = Path("/Users/joregan/Playing/hsi_ctmedit")

for ctmfile in CTMDIR.glob("*.txt"):
    ctmlines = readctm(str(ctmfile))
    outlines = []
    for ctmline in ctmlines:
        if len(ctmline) < 7:
            outlines.append(ctmline)
            continue
        if ctmline[7] != "ins":
            outlines.append(ctmline)
        else:
            if ctmline[4] in ["ah", "eh", "oh", "mhm", "m", "um"]:
                ctmline[7] = "cor"
                ctmline[6] = f"[{ctmline[4]}]"
            outlines.append(ctmline)
    with open(str(ctmfile), "w") as outf:
        for line in outlines:
            outf.write(" ".join(line) + "\n")

In [90]:
import json
from pathlib import Path
def json_ctm(filename, edit = False):
    with open(filename) as inf:
        data = json.load(inf)
        fileid = Path(filename).stem.replace(".w2v", "")
        for chunk in data['chunks']:
            start = chunk['timestamp'][0]
            end = chunk['timestamp'][1]
            word = chunk['text'].lower()
            if edit:
                edit = f" {word} cor"
            else:
                edit = ""
            print(f"{fileid} 1 {start} {end} {word} 1.0{edit}")

In [ ]:
json_ctm('/Users/joregan/Playing/hsi/audio/wav2vec/hsi_7_0719_222_003_main.w2v.json')

In [88]:
def json_ctm_path(fid, ctmedit=True):
    return json_ctm(f"/Users/joregan/Playing/hsi/audio/wav2vec/{fid}.w2v.json", ctmedit)

In [ ]:
json_ctm_path("hsi_7_0719_222_001_inter", True)